In [14]:
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

In [15]:
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,)),
])

train_fashion = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_fashion = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

def keep_first_five_classes(dataset):
    indices = [i for i, (_, label) in enumerate(dataset) if label < 5]
    return Subset(dataset, indices)

train_fashion_5 = keep_first_five_classes(train_fashion)
test_fashion_5 = keep_first_five_classes(test_fashion)

print(f"Обучающих примеров: {len(train_fashion_5)}")
print(f"Тестовых примеров: {len(test_fashion_5)}")

train_loader = DataLoader(train_fashion_5, batch_size=64, shuffle=True)
test_loader = DataLoader(test_fashion_5, batch_size=64)

Обучающих примеров: 30000
Тестовых примеров: 5000


In [22]:
class MLP(nn.Module):
    def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(28*28, 512)
            self.bn1 = nn.BatchNorm1d(512)
            self.fc2 = nn.Linear(512, 1024)
            self.bn2 = nn.BatchNorm1d(1024)
            self.fc3 = nn.Linear(1024, 512)
            self.bn3 = nn.BatchNorm1d(512)
            self.fc4 = nn.Linear(512, 256)
            self.bn4 = nn.BatchNorm1d(256)
            self.fc5 = nn.Linear(256, 10)
            self.dropout = nn.Dropout(0.1)
        
    def forward(self, x):
        x = x.flatten(1)
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = F.relu(self.bn3(self.fc3(x)))
        x = self.dropout(x)
        x = F.relu(self.bn4(self.fc4(x)))
        x = self.dropout(x)
        return self.fc5(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP().to(device)
print(device)

try:
    model.load_state_dict(torch.load("mnist_mlp.pt", map_location=device))
    print("Модель загружена успешно!")
except:
    print("Файл mnist_mlp.pt не найден. Обучите модель сначала.")

cuda
Модель загружена успешно!


In [23]:

model.fc5 = nn.Sequential(
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(128, 5)
).to(device)


for param in model.parameters():
    param.requires_grad = False

for param in model.fc5.parameters():
    param.requires_grad = True

In [24]:
def train_epoch(model, device, loader, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
    
    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc

def test_model(model, device, loader):
    model.eval()
    correct = 0
    
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
    
    return correct / len(loader.dataset)

In [25]:
optimizer = torch.optim.Adam(model.fc5.parameters(), lr=1e-3)

epochs = 10
history_transfer = {'train_acc': [], 'test_acc': [], 'loss': []}

for epoch in range(1, epochs + 1):
    epoch_start = time.time()
    loss, train_acc = train_epoch(model, device, train_loader, optimizer)
    test_acc = test_model(model, device, test_loader)
    
    history_transfer['train_acc'].append(train_acc)
    history_transfer['test_acc'].append(test_acc)
    history_transfer['loss'].append(loss)
    
    print(f"Эпоха {epoch:2d} | Потери: {loss:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f} | Время: {time.time() - epoch_start:.2f}с")

Эпоха  1 | Потери: 0.8151 | Train: 0.6846 | Test: 0.7442 | Время: 9.06с
Эпоха  2 | Потери: 0.7100 | Train: 0.7245 | Test: 0.7590 | Время: 9.02с
Эпоха  3 | Потери: 0.6829 | Train: 0.7349 | Test: 0.7628 | Время: 9.03с
Эпоха  4 | Потери: 0.6662 | Train: 0.7408 | Test: 0.7684 | Время: 8.96с
Эпоха  5 | Потери: 0.6569 | Train: 0.7461 | Test: 0.7774 | Время: 8.98с
Эпоха  6 | Потери: 0.6528 | Train: 0.7462 | Test: 0.7768 | Время: 9.03с
Эпоха  7 | Потери: 0.6425 | Train: 0.7488 | Test: 0.7818 | Время: 9.02с
Эпоха  8 | Потери: 0.6427 | Train: 0.7502 | Test: 0.7850 | Время: 9.03с
Эпоха  9 | Потери: 0.6327 | Train: 0.7520 | Test: 0.7822 | Время: 8.99с
Эпоха 10 | Потери: 0.6278 | Train: 0.7546 | Test: 0.7900 | Время: 9.02с


## Дообучение всей модели

In [20]:
for param in model.parameters():
    param.requires_grad = True


optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [21]:
for epoch in range(1, 11):
    epoch_start = time.time()
    loss, train_acc = train_epoch(model, device, train_loader, optimizer)
    test_acc = test_model(model, device, test_loader)
    print(f"Эпоха {epoch:2d} | Loss: {loss:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f} | {time.time() - epoch_start:.2f}с")

Эпоха  1 | Loss: 0.5014 | Train: 0.8133 | Test: 0.8632 | 9.55с
Эпоха  2 | Loss: 0.3841 | Train: 0.8601 | Test: 0.8780 | 10.20с
Эпоха  3 | Loss: 0.3393 | Train: 0.8789 | Test: 0.8866 | 10.63с
Эпоха  4 | Loss: 0.3147 | Train: 0.8860 | Test: 0.8926 | 10.28с
Эпоха  5 | Loss: 0.2922 | Train: 0.8966 | Test: 0.8972 | 10.24с
Эпоха  6 | Loss: 0.2791 | Train: 0.8992 | Test: 0.8970 | 10.16с
Эпоха  7 | Loss: 0.2677 | Train: 0.9044 | Test: 0.8994 | 10.54с
Эпоха  8 | Loss: 0.2585 | Train: 0.9077 | Test: 0.9046 | 9.91с
Эпоха  9 | Loss: 0.2491 | Train: 0.9086 | Test: 0.9048 | 10.12с
Эпоха 10 | Loss: 0.2391 | Train: 0.9141 | Test: 0.9076 | 10.52с


## Обучение на одежде 

In [26]:
class FashionMLP(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 1024)
        self.bn2 = nn.BatchNorm1d(1024)
        self.fc3 = nn.Linear(1024, 512)
        self.bn3 = nn.BatchNorm1d(512)
        self.fc4 = nn.Linear(512, 256)
        self.bn4 = nn.BatchNorm1d(256)
        self.fc5 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        x = x.flatten(1)
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = F.relu(self.bn3(self.fc3(x)))
        x = self.dropout(x)
        x = F.relu(self.bn4(self.fc4(x)))
        x = self.dropout(x)
        return self.fc5(x)

def train_epoch(model, device, loader, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def test_model(model, device, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
    return correct / len(loader.dataset)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [32]:
model = FashionMLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(1, 11):
    loss, train_acc = train_epoch(model, device, train_loader, optimizer)
    test_acc = test_model(model, device, test_loader)
    print(f"Эпоха {epoch:2d} | Loss: {loss:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")

torch.save(model.state_dict(), "fashion5_model.pt")
print("Модель сохранена как 'fashion5_model.pt'")

Эпоха  1 | Loss: 0.3875 | Train: 0.8599 | Test: 0.8834
Эпоха  2 | Loss: 0.3145 | Train: 0.8879 | Test: 0.8878
Эпоха  3 | Loss: 0.2860 | Train: 0.8962 | Test: 0.8856
Эпоха  4 | Loss: 0.2674 | Train: 0.9040 | Test: 0.8980
Эпоха  5 | Loss: 0.2544 | Train: 0.9073 | Test: 0.8822
Эпоха  6 | Loss: 0.2414 | Train: 0.9117 | Test: 0.9024
Эпоха  7 | Loss: 0.2283 | Train: 0.9157 | Test: 0.9064
Эпоха  8 | Loss: 0.2209 | Train: 0.9185 | Test: 0.9062
Эпоха  9 | Loss: 0.2101 | Train: 0.9221 | Test: 0.9098
Эпоха 10 | Loss: 0.2045 | Train: 0.9251 | Test: 0.9084
Модель сохранена как 'fashion5_model.pt'


In [33]:
model = FashionMLP().to(device)
model.load_state_dict(torch.load("fashion5_model.pt", map_location=device))
print("Модель загружена")

model.fc5 = nn.Linear(256, 10).to(device)

for param in model.parameters():
    param.requires_grad = False

for param in model.fc5.parameters():
    param.requires_grad = True



Модель загружена


In [34]:
train_mnist = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_mnist = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader_mnist = DataLoader(train_mnist, batch_size=64, shuffle=True)
test_loader_mnist = DataLoader(test_mnist, batch_size=64)


In [ ]:
optimizer = torch.optim.Adam(model.fc5.parameters(), lr=1e-3)

for epoch in range(1, 11):
    loss, train_acc = train_epoch(model, device, train_loader_mnist, optimizer)
    test_acc = test_model(model, device, test_loader_mnist)
    print(f"Эпоха {epoch:2d} | Loss: {loss:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")

Эпоха  1 | Loss: 1.6992 | Train: 0.4097 | Test: 0.5221
Эпоха  2 | Loss: 1.5939 | Train: 0.4496 | Test: 0.5414
Эпоха  3 | Loss: 1.5769 | Train: 0.4572 | Test: 0.5461
Эпоха  4 | Loss: 1.5695 | Train: 0.4590 | Test: 0.5492
Эпоха  5 | Loss: 1.5696 | Train: 0.4595 | Test: 0.5560
Эпоха  6 | Loss: 1.5636 | Train: 0.4612 | Test: 0.5474
Эпоха  7 | Loss: 1.5592 | Train: 0.4631 | Test: 0.5443
Эпоха  8 | Loss: 1.5673 | Train: 0.4602 | Test: 0.5422
Эпоха  9 | Loss: 1.5622 | Train: 0.4618 | Test: 0.5508
Эпоха 10 | Loss: 1.5577 | Train: 0.4623 | Test: 0.5498


In [36]:
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(1, 11):
    loss, train_acc = train_epoch(model, device, train_loader_mnist, optimizer)
    test_acc = test_model(model, device, test_loader_mnist)
    print(f"Эпоха {epoch:2d} | Loss: {loss:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")

Эпоха  1 | Loss: 0.7972 | Train: 0.7389 | Test: 0.8960
Эпоха  2 | Loss: 0.3729 | Train: 0.8875 | Test: 0.9375
Эпоха  3 | Loss: 0.2712 | Train: 0.9196 | Test: 0.9512
Эпоха  4 | Loss: 0.2221 | Train: 0.9335 | Test: 0.9601
Эпоха  5 | Loss: 0.1919 | Train: 0.9429 | Test: 0.9654
Эпоха  6 | Loss: 0.1661 | Train: 0.9498 | Test: 0.9681
Эпоха  7 | Loss: 0.1505 | Train: 0.9549 | Test: 0.9708
Эпоха  8 | Loss: 0.1386 | Train: 0.9581 | Test: 0.9731
Эпоха  9 | Loss: 0.1245 | Train: 0.9625 | Test: 0.9747
Эпоха 10 | Loss: 0.1158 | Train: 0.9641 | Test: 0.9754
